# Titanic Survival Prediction

This notebook builds a machine learning workflow to predict passenger survival.

The first part focuses on creating useful features from the raw passenger data. Later sections will cover preprocessing, model training, evaluation, and model interpretation.


## 1. Setup and Data Loading

First, I import the libraries I need and load the training and test datasets.


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import warnings
warnings.filterwarnings("ignore")


In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

print("Training data shape:", train.shape)
print("Test data shape:", test.shape)


Training data shape: (891, 12)
Test data shape: (418, 11)


## 2. Initial Data Check

Before creating features, I checked the main columns and the data types.


In [3]:
train.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
train.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


## 3. Feature Engineering

I will create a few features that may give the models more useful information than the raw columns alone.

The features are:

- `Title`
- `FamilySize`
- `IsAlone`
- `AgeBin`
- `FareBin`
- `Deck`
- `CabinKnown`


### Extract Title

The title is taken from the passenger's name.

Titles such as `Mr`, `Mrs`, `Miss`, and `Master` can give some information about a passenger's age and social group.


In [5]:
train["Title"] = train["Name"].str.extract(r",\s*([^.]*)\.", expand=False).str.strip()
test["Title"] = test["Name"].str.extract(r",\s*([^.]*)\.", expand=False).str.strip()

train["Title"].value_counts()


Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Mlle              2
Major             2
Col               2
the Countess      1
Capt              1
Ms                1
Sir               1
Lady              1
Mme               1
Don               1
Jonkheer          1
Name: count, dtype: int64

### Create FamilySize

`FamilySize` shows how many people were travelling together in the passenger's immediate family.

The passenger is included in the total.


In [6]:
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1

train["FamilySize"].describe()


count    891.000000
mean       1.904602
std        1.613459
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       11.000000
Name: FamilySize, dtype: float64

### Create IsAlone

`IsAlone` shows whether a passenger was travelling alone.

A value of `1` means the passenger was alone, while `0` means they were travelling with family.


In [7]:
train["IsAlone"] = (train["FamilySize"] == 1).astype(int)
test["IsAlone"] = (test["FamilySize"] == 1).astype(int)

train["IsAlone"].value_counts()


IsAlone
1    537
0    354
Name: count, dtype: int64

### Create AgeBin

Age is grouped into a few simple categories.

I am leaving missing ages as missing for now. They will be handled later by the preprocessing pipeline.


In [8]:
age_bins = [0, 12, 18, 35, 60, np.inf]
age_labels = ["Child", "Teen", "Young Adult", "Adult", "Senior"]

train["AgeBin"] = pd.cut(
    train["Age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

test["AgeBin"] = pd.cut(
    test["Age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

train["AgeBin"].value_counts(dropna=False)


AgeBin
Young Adult    358
Adult          195
NaN            177
Teen            70
Child           69
Senior          22
Name: count, dtype: int64

### Create FareBin

Fare is grouped into four categories using the quartiles of the training data.

The bin edges come from the training data so the test data does not influence them.


In [9]:
fare_bins = train["Fare"].quantile([0, 0.25, 0.50, 0.75, 1.00]).values
fare_bins = np.unique(fare_bins)

fare_labels = ["Low", "Medium", "High", "Very High"][:len(fare_bins) - 1]

train["FareBin"] = pd.cut(
    train["Fare"],
    bins=fare_bins,
    labels=fare_labels,
    include_lowest=True
)

test["FareBin"] = pd.cut(
    test["Fare"],
    bins=fare_bins,
    labels=fare_labels,
    include_lowest=True
)

train["FareBin"].value_counts(dropna=False)


FareBin
Medium       224
Low          223
High         222
Very High    222
Name: count, dtype: int64

### Extract Deck

The first character of `Cabin` is used as the deck.

Missing cabin information is labelled as `Unknown` instead of creating a made-up deck.


In [10]:
train["Deck"] = train["Cabin"].str[0].fillna("Unknown")
test["Deck"] = test["Cabin"].str[0].fillna("Unknown")

train["Deck"].value_counts()


Deck
Unknown    687
C           59
B           47
D           33
E           32
A           15
F           13
G            4
T            1
Name: count, dtype: int64

### Create CabinKnown

`CabinKnown` shows whether cabin information is available.

This keeps the information about missing cabin records without trying to guess the missing cabin.


In [11]:
train["CabinKnown"] = train["Cabin"].notna().astype(int)
test["CabinKnown"] = test["Cabin"].notna().astype(int)

train["CabinKnown"].value_counts()


CabinKnown
0    687
1    204
Name: count, dtype: int64

### Check the New Features

I will check the new columns before moving on to the preprocessing stage.


In [12]:
new_features = [
    "Title",
    "FamilySize",
    "IsAlone",
    "AgeBin",
    "FareBin",
    "Deck",
    "CabinKnown"
]

train[new_features].head()


,Title,FamilySize,IsAlone,AgeBin,FareBin,Deck,CabinKnown
0,Mr,2,0,Young Adult,Low,Unknown,0
1,Mrs,2,0,Adult,Very High,C,1
2,Miss,1,1,Young Adult,Medium,Unknown,0
3,Mrs,2,0,Young Adult,Very High,C,1
4,Mr,1,1,Young Adult,Medium,Unknown,0


In [13]:
train[new_features].info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   Title       891 non-null    object  
 1   FamilySize  891 non-null    int64   
 2   IsAlone     891 non-null    int32   
 3   AgeBin      714 non-null    category
 4   FareBin     891 non-null    category
 5   Deck        891 non-null    object  
 6   CabinKnown  891 non-null    int32   
dtypes: category(2), int32(2), int64(1), object(2)
memory usage: 30.1+ KB


In [14]:
train[new_features].isnull().sum()


Title           0
FamilySize      0
IsAlone         0
AgeBin        177
FareBin         0
Deck            0
CabinKnown      0
dtype: int64

## 4. Next Step

The feature engineering is now complete.

The next step is to split the training data and build a leakage-free preprocessing pipeline using `ColumnTransformer` and `Pipeline`.
